In [121]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    when,
    avg,
    count
)

from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

from pyspark.ml import Pipeline

from pyspark.ml.classification import (
    RandomForestClassifier
)

from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator
)

from pyspark.ml.functions import vector_to_array

# from pyspark.sql.functions import when
# from pyspark.sql.functions import col

In [122]:
spark = (
    SparkSession.builder
    .appName("Gaming Satisfaction Prediction")
    .config("spark.driver.memory","2g")
    .config("spark.executor.memory","2g")
    .config("spark.local.dir","D:/spark_temp")
    .getOrCreate()
)

In [123]:
print("\n========== LOAD GOLD DATA ==========\n")
fact = spark.read.parquet(
    "../data/gold/fact_game_analysis"
)

dim_game = spark.read.parquet(
    "../data/gold/dim_game"
)

dim_genre = spark.read.parquet(
    "../data/gold/dim_genre"
)

dim_price = spark.read.parquet(
    "../data/gold/dim_price"
)

fact.printSchema()
dim_game.printSchema()
dim_genre.printSchema()
dim_price.printSchema()


========== LOAD GOLD DATA ==========

root
 |-- game_key: integer (nullable = true)
 |-- app_id: integer (nullable = true)
 |-- genre_key: integer (nullable = true)
 |-- publisher_key: integer (nullable = true)
 |-- developer_key: integer (nullable = true)
 |-- price_key: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- average_playtime: integer (nullable = true)
 |-- positive_ratings: integer (nullable = true)
 |-- negative_ratings: integer (nullable = true)
 |-- total_reviews: long (nullable = true)
 |-- positive_reviews: long (nullable = true)
 |-- negative_reviews: long (nullable = true)
 |-- recommendation_rate: double (nullable = true)
 |-- average_review_score: double (nullable = true)
 |-- rawg_rating: double (nullable = true)
 |-- rawg_metacritic: integer (nullable = true)

root
 |-- game_key: integer (nullable = true)
 |-- app_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (

In [124]:
print("\n========== JOIN GOLD DATA ==========\n")
ml_data = (
    fact
    .join(
        dim_game,
        "game_key",
        "left"
    )
)

ml_data = (
    ml_data
    .join(
        dim_genre,
        "genre_key",
        "left"
    )
)

ml_data = (
    ml_data
    .join(
        dim_price,
        "price_key",
        "left"
    )
)

ml_data.printSchema()


========== JOIN GOLD DATA ==========

root
 |-- price_key: integer (nullable = true)
 |-- genre_key: integer (nullable = true)
 |-- game_key: integer (nullable = true)
 |-- app_id: integer (nullable = true)
 |-- publisher_key: integer (nullable = true)
 |-- developer_key: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- average_playtime: integer (nullable = true)
 |-- positive_ratings: integer (nullable = true)
 |-- negative_ratings: integer (nullable = true)
 |-- total_reviews: long (nullable = true)
 |-- positive_reviews: long (nullable = true)
 |-- negative_reviews: long (nullable = true)
 |-- recommendation_rate: double (nullable = true)
 |-- average_review_score: double (nullable = true)
 |-- rawg_rating: double (nullable = true)
 |-- rawg_metacritic: integer (nullable = true)
 |-- app_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genres: string (nullable 

In [125]:
print("\n========== CREATE SATISFACTION LABEL ==========\n")
ml_data = ml_data.withColumn(
    "satisfaction_label",
    when(
        col("recommendation_rate") >= 0.7,
        1
    )
    .otherwise(0)
)

ml_data.groupBy(
    "satisfaction_label"
).count().show()


========== CREATE SATISFACTION LABEL ==========

+------------------+-----+
|satisfaction_label|count|
+------------------+-----+
|                 1|14066|
|                 0|62396|
+------------------+-----+



In [126]:
print("\n========== SELECT MODEL FEATURES ==========\n")
model_data = ml_data.select(
    "name",
    "genre",
    "price_category",

    "recommendation_rate",

    "price",
    "average_playtime",
    "positive_reviews",
    "negative_reviews",
    "rawg_rating",
    "rawg_metacritic",

    "satisfaction_label"
)


========== SELECT MODEL FEATURES ==========



In [127]:
print("\n========== FILL MISSING VALUES ==========\n")
model_data = model_data.fillna(
    {
        "rawg_rating":0,
        "rawg_metacritic":0
    }
)


========== FILL MISSING VALUES ==========



In [128]:
print("\n========== ENCODE CATEGORICAL VARIABLES ==========\n")
genre_indexer = StringIndexer(
    inputCol="genre",
    outputCol="genre_index",
    handleInvalid="keep"
)

price_indexer = StringIndexer(
    inputCol="price_category",
    outputCol="price_index",
    handleInvalid="keep"
)

encoder = OneHotEncoder(
    inputCols=[
        "genre_index",
        "price_index"
    ],
    outputCols=[
        "genre_vector",
        "price_vector"
    ]
)


========== ENCODE CATEGORICAL VARIABLES ==========



In [129]:
print("\n========== COMBINE FEATURES ==========\n")
assembler = VectorAssembler(
    inputCols=[
        "price",
        "average_playtime",
        "positive_reviews",
        "negative_reviews",
        "rawg_rating",
        "rawg_metacritic",
        "genre_vector",
        "price_vector"
    ],
    outputCol="features"
)


========== COMBINE FEATURES ==========



In [130]:
print("\n========== RANDOM FOREST MODEL ==========\n")
rf = RandomForestClassifier(
    labelCol="satisfaction_label",
    featuresCol="features",
    numTrees=100
)


========== RANDOM FOREST MODEL ==========



In [131]:
print("\n========== ML PIPELINE ==========\n")
pipeline = Pipeline(
    stages=[
        genre_indexer,
        price_indexer,
        encoder,
        assembler,
        rf
    ]
)


========== ML PIPELINE ==========



In [132]:
print("\n========== SPLIT DATA ==========\n")
train_data, test_data = model_data.randomSplit(
    [0.8,0.2],
    seed=42
)

print(train_data.count())
print(test_data.count())


========== SPLIT DATA ==========

60983
15479


In [133]:
print("\n========== TRAIN MODEL ==========\n")
model = pipeline.fit(
    train_data
)


========== TRAIN MODEL ==========



In [134]:
print("\n========== PREDICTION ==========\n")
prediction = model.transform(
    test_data
)

prediction = prediction.withColumn(
    "prediction_result",
    when(
        col("prediction") == 1,
        "Satisfied"
    )
    .otherwise(
        "Not Satisfied"
    )
)

prediction.select(
    "name",
    "satisfaction_label",
    "prediction",
    "probability",
    "prediction_result"
).show(20,False)


========== PREDICTION ==========

+--------------------------------------------+------------------+----------+-----------------------------------------+-----------------+
|name                                        |satisfaction_label|prediction|probability                              |prediction_result|
+--------------------------------------------+------------------+----------+-----------------------------------------+-----------------+
|! That Bastard Is Trying To Steal Our Gold !|0                 |0.0       |[0.574835372409649,0.425164627590351]    |Not Satisfied    |
|!LABrpgUP!                                  |0                 |0.0       |[0.9515973573600137,0.048402642639986344]|Not Satisfied    |
|!LABrpgUP!                                  |0                 |0.0       |[0.9504946855867361,0.049505314413263904]|Not Satisfied    |
|"""Glow Ball"" - The billiard puzzle game"  |0                 |0.0       |[0.5876847714392516,0.41231522856074837] |Not Satisfied    |
|"Abha

In [135]:
print("\n========== EVALUATE MODEL ==========\n")
evaluator = MulticlassClassificationEvaluator(
    labelCol="satisfaction_label",
    predictionCol="prediction",
    metricName="accuracy"
)


accuracy = evaluator.evaluate(
    prediction
)

print(
    "Accuracy:",
    accuracy
)


========== EVALUATE MODEL ==========

Accuracy: 0.8541895471283675


In [136]:
print("\n========== SAVE PREDICTION RESULTS ==========\n")

result = prediction.withColumn(
    "probability_array",
    vector_to_array(col("probability"))
)

result = result.select(
    "name",
    "genre",
    "price_category",
    "recommendation_rate",
    "prediction",
    col("probability_array")[0].alias("not_satisfied_probability"),
    col("probability_array")[1].alias("satisfied_probability"),
    "prediction_result"
)


result.show(5, truncate=False)

result.write \
.mode("overwrite") \
.parquet(
    "../data/gold/game_prediction"
)



========== SAVE PREDICTION RESULTS ==========

+--------------------------------------------+---------+--------------+-------------------+----------+-------------------------+---------------------+-----------------+
|name                                        |genre    |price_category|recommendation_rate|prediction|not_satisfied_probability|satisfied_probability|prediction_result|
+--------------------------------------------+---------+--------------+-------------------+----------+-------------------------+---------------------+-----------------+
|! That Bastard Is Trying To Steal Our Gold !|Casual   |Low Price     |0.32               |0.0       |0.574835372409649        |0.425164627590351    |Not Satisfied    |
|!LABrpgUP!                                  |Adventure|Low Price     |0.0                |0.0       |0.9515973573600137       |0.048402642639986344 |Not Satisfied    |
|!LABrpgUP!                                  |Indie    |Low Price     |0.0                |0.0       |0.950

In [137]:
result.show(5, truncate=False)
prediction.printSchema()
prediction.show(30, truncate=False)

+--------------------------------------------+---------+--------------+-------------------+----------+-------------------------+---------------------+-----------------+
|name                                        |genre    |price_category|recommendation_rate|prediction|not_satisfied_probability|satisfied_probability|prediction_result|
+--------------------------------------------+---------+--------------+-------------------+----------+-------------------------+---------------------+-----------------+
|! That Bastard Is Trying To Steal Our Gold !|Casual   |Low Price     |0.32               |0.0       |0.574835372409649        |0.425164627590351    |Not Satisfied    |
|!LABrpgUP!                                  |Adventure|Low Price     |0.0                |0.0       |0.9515973573600137       |0.048402642639986344 |Not Satisfied    |
|!LABrpgUP!                                  |Indie    |Low Price     |0.0                |0.0       |0.9504946855867361       |0.049505314413263904 |Not S